# Model Training
This notebook guides you through training the XGBoost tabular model and the 1D ResNet ECG CNN model.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_dataset import load_or_create_datasets
from src.data.preprocessing import preprocess_ecg_data, preprocess_tabular_data
from src.models.cnn_ecg import ECGCNN, train_ecg_cnn
from src.models.xgboost_model import XGBoostModelWrapper
from src.utils.config import Config
from src.utils.metrics import evaluate_predictions

config = Config(str(PROJECT_ROOT / "configs/params.yaml"), str(PROJECT_ROOT / "configs/paths.yaml"))
random_state = config.get("preprocessing", {}).get("random_state", 42)
np.random.seed(random_state)
torch.manual_seed(random_state)

In [ ]:
df, ecg = load_or_create_datasets(config)
X_train, X_test, y_train, y_test, preprocessor = preprocess_tabular_data(df, config)
processed_ecg = preprocess_ecg_data(ecg, config)
train_indices = X_train.index.to_numpy()
test_indices = X_test.index.to_numpy()
X_ecg_train = processed_ecg[train_indices]
X_ecg_test = processed_ecg[test_indices]
y_ecg_train = y_train.to_numpy()
y_ecg_test = y_test.to_numpy()
print(f"Train/test tabular shapes: {X_train.shape}, {X_test.shape}")
print(f"Train/test ECG shapes: {X_ecg_train.shape}, {X_ecg_test.shape}")

In [ ]:
xgb_wrapper = XGBoostModelWrapper(config)
xgb_wrapper.train(X_train, y_train)
xgb_wrapper.save()
xgb_predictions = xgb_wrapper.predict(X_test)
xgb_probabilities = xgb_wrapper.predict_proba(X_test)[:, 1]
xgb_metrics = evaluate_predictions(y_test, xgb_predictions, xgb_probabilities)
print("XGBoost metrics:")
print(pd.Series(xgb_metrics).drop("confusion_matrix"))

In [ ]:
cnn_model = ECGCNN(in_channels=X_ecg_train.shape[1], num_classes=2)
train_ecg_cnn(cnn_model, X_ecg_train, y_ecg_train, config)

cnn_model.eval()
with torch.no_grad():
    cnn_logits = cnn_model(torch.tensor(X_ecg_test, dtype=torch.float32))
cnn_predictions = cnn_logits.argmax(dim=1).cpu().numpy()
cnn_probabilities = torch.softmax(cnn_logits, dim=1)[:, 1].cpu().numpy()
cnn_metrics = evaluate_predictions(y_ecg_test, cnn_predictions, cnn_probabilities)
print("ECG CNN metrics:")
print(pd.Series(cnn_metrics).drop("confusion_matrix"))

In [ ]:
metrics_table = pd.DataFrame([xgb_metrics, cnn_metrics], index=["XGBoost", "ECG CNN"])
display(metrics_table.drop(columns="confusion_matrix"))